In [1]:
import json

with open('example.json') as f:
    d = json.load(f)

In [2]:
import sys
import os
from pathlib import Path

root_dir = Path(os.getcwd()).parent / 'app'
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

In [3]:
from db.db import get_session, init_db

init_db()

#with get_session() as session:

In [4]:
class AutoDBDict:
    def __init__(self) -> None:
        self.stmts = {}
    
    def set_stmt(self, object_type, stmt):
        self.stmts[object_type] = stmt

    def add_to_db(self, object, session):
        db_object = session.exec(self.stmts[type(object)](object)).first()
        if not db_object:
            db_object = object
            session.add(db_object)
            session.flush()
        return db_object

to_db = AutoDBDict()

In [5]:
from models.product import Product
from models.attribute import Attribute
from models.attribute_name import AttributeName
from models.property_name import PropertyName
from models.property import Property
from models.variant import Variant
from sqlmodel import select

to_db.set_stmt(Product, lambda obj: select(Product).where(Product.name == obj.name))
to_db.set_stmt(AttributeName, lambda obj: select(AttributeName).where(AttributeName.name == obj.name))
to_db.set_stmt(Attribute, lambda obj: select(Attribute).where(Attribute.attribute_name_id == obj.attribute_name_id, Attribute.value == obj.value))
to_db.set_stmt(PropertyName, lambda obj: select(PropertyName).where(PropertyName.name == obj.name))
to_db.set_stmt(Property, lambda obj: select(Property).where(Property.property_name_id == obj.property_name_id, Property.value == obj.value, Property.attribute_id == obj.attribute_id))
to_db.set_stmt(Variant, lambda obj: select(Variant).where(Variant.product == obj.product))

id_to_att_prop = {}

with get_session() as session:
    product_db = to_db.add_to_db(Product(name=d['data']['result']['GLOBAL_DATA']['globalData']['subject']), session)
    
    for att in d['data']['result']['PRODUCT_PROP_PC']['showedProps']:
        an = to_db.add_to_db(AttributeName(name=att['attrName']), session)
        to_db.add_to_db(Attribute(attribute_name_id=an.id, value=att['attrValue']), session)

    for prop in d['data']['result']['SKU']['skuProperties']:
        an = to_db.add_to_db(AttributeName(name=prop['skuPropertyName']), session)
        for val in prop['skuPropertyValues']:
            a = to_db.add_to_db(Attribute(attribute_name_id=an.id, value=val['propertyValueName']), session)
            id_to_att_prop[f'{prop['skuPropertyId']}:{val['propertyValueIdLong']}'] = (a, [])
            if "propertySizeChartInfo" in val:
                for p in val["propertySizeChartInfo"]:
                    pn = to_db.add_to_db(PropertyName(name=p['name']), session)
                    id_to_att_prop[f'{prop['skuPropertyId']}:{val['propertyValueIdLong']}'][1].append((pn.id, p['value']))
    
    variants = {}

    for key, val in d['data']['result']['PRICE']['skuIdStrPriceInfoMap'].items():
        variants[key] = {"price": float(val['salePriceString'][:-2].replace(" ", "").replace(',', '.'))}

    for path in d['data']['result']['SKU']['skuPaths']:
        variants[path['skuIdStr']]['attributes'] = []
        att_combination = path['path'].split(";")
        for seq in att_combination:
            variants[path['skuIdStr']]['attributes'].append(id_to_att_prop[seq][0])

        var = to_db.add_to_db(Variant(product_id=product_db.id, price=int(variants[path['skuIdStr']]['price']), stock=path['skuStock'], attributes=variants[path['skuIdStr']]['attributes']), session)

        for seq in att_combination:
            if len(id_to_att_prop[seq][1]) != 0:
                for prop in id_to_att_prop[seq][1]:
                    to_db.add_to_db(Property(property_name_id=prop[0], attribute_id=id_to_att_prop[seq][0].id, variant_id=var.id, value=prop[1]), session)
                
        print(f"Variant: {var}")

    session.commit()

C:\Users\KK\AppData\Local\Temp\ipykernel_29380\532125467.py:19: SAWarning: relationship 'Order.variants' will copy column variant.id to column ordervariantlink.variant_id, which conflicts with relationship(s): 'OrderVariantLink.variant' (copies variant.id to ordervariantlink.variant_id). If this is not the intention, consider if these relationships should be linked with back_populates, or if viewonly=True should be applied to one or more if they are read-only. For the less common case that foreign key constraints are partially overlapping, the orm.foreign() annotation can be used to isolate the columns that should be written towards.   To silence this warning, add the parameter 'overlaps="variant"' to the 'Order.variants' relationship. (Background on this warning at: https://sqlalche.me/e/20/qzyx) (This warning originated from the `configure_mappers()` process, which was invoked automatically in response to a user-initiated operation.)
  product_db = to_db.add_to_db(Product(name=d['dat

Variant: price=17 stock=1986 product_id=16 id=118
Variant: price=34 stock=1971 product_id=16 id=119
Variant: price=28 stock=1979 product_id=16 id=120
Variant: price=28 stock=1984 product_id=16 id=121
Variant: price=23 stock=1982 product_id=16 id=122
Variant: price=27 stock=1973 product_id=16 id=123
Variant: price=13 stock=1978 product_id=16 id=124
Variant: price=39 stock=1984 product_id=16 id=125
Variant: price=29 stock=1983 product_id=16 id=126
Variant: price=13 stock=1964 product_id=16 id=127
Variant: price=12 stock=1983 product_id=16 id=128
Variant: price=34 stock=1987 product_id=16 id=129
Variant: price=31 stock=1988 product_id=16 id=130
Variant: price=30 stock=1984 product_id=16 id=131
Variant: price=28 stock=1980 product_id=16 id=132
Variant: price=23 stock=493 product_id=16 id=133
Variant: price=25 stock=1977 product_id=16 id=134
Variant: price=23 stock=1961 product_id=16 id=135
Variant: price=16 stock=7 product_id=16 id=136
Variant: price=32 stock=1979 product_id=16 id=137
Vari